# Daily Practice — 2026-09-18: Pandas / Data Wrangling — Building a Data Quality Gate

**Dataset:** [Titanic passenger manifest](https://github.com/mwaskom/seaborn-data) (891 real passengers, fetched live from seaborn's public dataset repo) — a genuinely messy real-world table: missing values, redundant/derived columns, mixed types, and a couple of dirty rows.

**Problem statement.** Imagine you've just moved from a QA role onto a team that trains a model on this data. Nobody wants to debug a silently-corrupted training set three sprints from now, so your first job is not modelling at all — it's building the **data quality gate** the pipeline runs before any training job is allowed to touch the data. That means: a reusable, declarative schema you can validate any incoming batch against, null-rate thresholds that distinguish "expected missingness" from "something is broken," a duplicate-row check, and a cleaning step whose *output* must itself pass the same gate before it's allowed downstream.

**What you should produce**, in order:
1. `load_data()` — loads the raw CSV into a `DataFrame`.
2. `validate_schema(df, schema)` — checks every column against a declarative schema (dtype family, allowed value set / numeric range) and returns a list of violation records (not just a pass/fail bool — you need to know *what* broke).
3. `check_null_rates(df, thresholds)` — flags any column whose null rate exceeds a per-column allowed threshold.
4. `check_duplicates(df, subset)` — returns the number of duplicate rows on a given key subset. This dataset has **no unique passenger ID**, so some duplication is expected coincidence (e.g. two unrelated 3rd-class passengers with identical fare/age/family-size) rather than a data error — treat it as an *advisory* signal, not a blocking one.
5. `run_quality_gate(df, schema, null_thresholds, dup_subset)` — combines 2–4 into one report. `passed` is `True` only when `schema_violations` and `null_violations` are both empty; `duplicate_count` is reported for visibility but does **not** affect `passed` (a real pipeline would route it to a human, not auto-block on it). This mirrors an SQA distinction you already know: not every failed check is the same severity — blocking vs. advisory.
6. `clean_data(df)` — fixes what's fixable (impute `age` by `pclass`+`sex` median, drop the unusably sparse `deck` column, drop the 2 rows missing `embark_town`) and returns `(cleaned_df, action_log)`.
7. A final cell that runs the gate on the **raw** data (expect failures), cleans it, runs the gate again on the **cleaned** data (expect a pass), and prints a before/after summary table.

This is the kind of artifact an SQA background makes you good at: don't just eyeball `.info()` and move on — turn "is this data okay" into a specific, re-runnable, falsifiable check.

## Setup

In [1]:
import pandas as pd
import numpy as np

TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"


## 1. Load the data

TODO: implement `load_data()`. It should read `TITANIC_URL` into a `DataFrame` and return it.

In [5]:
def load_data() -> pd.DataFrame:
    """Load the raw Titanic passenger manifest."""
    DataFrame = pd.read_csv(TITANIC_URL)
    return DataFrame
    # TODO: read TITANIC_URL with pandas and return the DataFrame

raw_df = load_data()
raw_df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Declarative schema + `validate_schema`

Below is a starter schema covering the columns that matter for a model that predicts `survived` from `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, and `embarked`. Each entry describes what a *valid* value looks like.

TODO: implement `validate_schema(df, schema)`. For every column in `schema`:
- if the column is missing from `df` entirely, record a violation with `check="missing_column"`.
- otherwise, for a column with an `allowed` set, flag any non-null value not in that set.
- for a column with a `min`/`max` range, flag any non-null value outside `[min, max]`.
- nulls are **not** violations here — that's what `check_null_rates` is for.

Return a `list[dict]`, one dict per violation, each with at least `column`, `check`, and `n_violations` keys (you decide the rest — e.g. a few example bad values is useful for debugging).

In [ ]:
SCHEMA = {
    "survived":  {"allowed": {0, 1}},
    "pclass":    {"allowed": {1, 2, 3}},
    "sex":       {"allowed": {"male", "female"}},
    "age":       {"min": 0, "max": 100},
    "sibsp":     {"min": 0, "max": 10},
    "parch":     {"min": 0, "max": 10},
    "fare":      {"min": 0, "max": 600},
    "embarked":  {"allowed": {"S", "C", "Q"}},
}

def validate_schema(df: pd.DataFrame, schema: dict) -> list:
    """Check df against a declarative schema, return a list of violation records."""
    # TODO
    raise NotImplementedError


## 3. Null-rate thresholds

TODO: implement `check_null_rates(df, thresholds)`. `thresholds` maps column name -> max allowed null *fraction* (0.0-1.0). Return a `list[dict]` of violations (only for columns that exceed their threshold), each with `column`, `null_rate`, `threshold`.

In [ ]:
NULL_THRESHOLDS = {
    "age": 0.25,        # some missing ages are expected, but not more than a quarter
    "embarked": 0.01,   # this should basically never be missing
    "deck": 0.5,        # deck is known to be very sparse; still worth a ceiling
}

def check_null_rates(df: pd.DataFrame, thresholds: dict) -> list:
    """Flag columns whose null rate exceeds their allowed threshold."""
    # TODO
    raise NotImplementedError


## 4. Duplicate rows

TODO: implement `check_duplicates(df, subset)`. Return the integer count of duplicate rows when comparing on `subset` (a list of column names), keeping the first occurrence as non-duplicate.

In [ ]:
DUP_SUBSET = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]

def check_duplicates(df: pd.DataFrame, subset: list) -> int:
    """Count duplicate rows on the given subset of columns."""
    # TODO
    raise NotImplementedError


## 5. The gate: combine everything into one report

TODO: implement `run_quality_gate(df, schema, null_thresholds, dup_subset)`. It should call the three checks above and return a dict:

```python
{
    "schema_violations": [...],
    "null_violations": [...],
    "duplicate_count": <int>,
    "passed": <bool>,   # True iff schema_violations and null_violations are both empty
                        # (duplicate_count is advisory only, see note above)
}
```

In [ ]:
def run_quality_gate(df: pd.DataFrame, schema: dict, null_thresholds: dict, dup_subset: list) -> dict:
    """Run the full data quality gate and return a combined report."""
    # TODO
    raise NotImplementedError


## 6. Cleaning

TODO: implement `clean_data(df)`. It should return `(cleaned_df, action_log)` where `action_log` is a `list[str]` describing what was done. Required fixes:
- Impute missing `age` with the **median age within the same `(pclass, sex)` group** (not a single global median).
- Drop the `deck` column entirely (it's missing on ~77% of rows — not salvageable).
- Drop rows where `embark_town` (and therefore `embarked`) is missing (there are only 2 such rows).
- Leave every other column untouched.

In [ ]:
def clean_data(df: pd.DataFrame):
    """Apply the fixable cleaning steps and return (cleaned_df, action_log)."""
    # TODO
    raise NotImplementedError


## 7. Put it together: raw vs. cleaned

TODO:
1. Run `run_quality_gate` on `raw_df`. It should **fail** (print why).
2. Run `clean_data` on `raw_df` to get `cleaned_df` and the action log. Print the log.
3. Run `run_quality_gate` on `cleaned_df` against the same schema/thresholds (drop `deck` from `NULL_THRESHOLDS` since the column no longer exists). It should **pass**.
4. Print a small before/after summary: row count, null counts per column, and the gate's `passed` value, for both `raw_df` and `cleaned_df`.

In [ ]:
# TODO: put it all together here


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import pandas as pd
import numpy as np

TITANIC_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"


def load_data() -> pd.DataFrame:
    return pd.read_csv(TITANIC_URL)


SCHEMA = {
    "survived":  {"allowed": {0, 1}},
    "pclass":    {"allowed": {1, 2, 3}},
    "sex":       {"allowed": {"male", "female"}},
    "age":       {"min": 0, "max": 100},
    "sibsp":     {"min": 0, "max": 10},
    "parch":     {"min": 0, "max": 10},
    "fare":      {"min": 0, "max": 600},
    "embarked":  {"allowed": {"S", "C", "Q"}},
}


def validate_schema(df: pd.DataFrame, schema: dict) -> list:
    violations = []
    for col, rules in schema.items():
        if col not in df.columns:
            violations.append({"column": col, "check": "missing_column", "n_violations": None})
            continue

        series = df[col].dropna()

        if "allowed" in rules:
            bad = series[~series.isin(rules["allowed"])]
            if len(bad):
                violations.append({
                    "column": col, "check": "allowed_values",
                    "n_violations": len(bad), "examples": bad.unique()[:5].tolist(),
                })

        if "min" in rules or "max" in rules:
            lo = rules.get("min", -np.inf)
            hi = rules.get("max", np.inf)
            bad = series[(series < lo) | (series > hi)]
            if len(bad):
                violations.append({
                    "column": col, "check": "range",
                    "n_violations": len(bad), "examples": bad.unique()[:5].tolist(),
                })

    return violations


NULL_THRESHOLDS = {"age": 0.25, "embarked": 0.01, "deck": 0.5}


def check_null_rates(df: pd.DataFrame, thresholds: dict) -> list:
    violations = []
    for col, max_rate in thresholds.items():
        if col not in df.columns:
            continue
        null_rate = df[col].isna().mean()
        if null_rate > max_rate:
            violations.append({"column": col, "null_rate": round(null_rate, 4), "threshold": max_rate})
    return violations


DUP_SUBSET = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]


def check_duplicates(df: pd.DataFrame, subset: list) -> int:
    return int(df.duplicated(subset=subset, keep="first").sum())


def run_quality_gate(df: pd.DataFrame, schema: dict, null_thresholds: dict, dup_subset: list) -> dict:
    schema_violations = validate_schema(df, schema)
    null_violations = check_null_rates(df, null_thresholds)
    duplicate_count = check_duplicates(df, dup_subset)  # advisory only, not gating
    passed = not schema_violations and not null_violations
    return {
        "schema_violations": schema_violations,
        "null_violations": null_violations,
        "duplicate_count": duplicate_count,
        "passed": passed,
    }


def clean_data(df: pd.DataFrame):
    df = df.copy()
    log = []

    before = df["age"].isna().sum()
    df["age"] = df.groupby(["pclass", "sex"])["age"].transform(lambda s: s.fillna(s.median()))
    log.append(f"Imputed {before} missing 'age' values with (pclass, sex) group median")

    df = df.drop(columns=["deck"])
    log.append("Dropped 'deck' column (~77% null, not salvageable)")

    before_n = len(df)
    df = df.dropna(subset=["embark_town"])
    log.append(f"Dropped {before_n - len(df)} rows missing 'embark_town'/'embarked'")

    return df, log


raw_df = load_data()

raw_report = run_quality_gate(raw_df, SCHEMA, NULL_THRESHOLDS, DUP_SUBSET)
print("RAW gate passed:", raw_report["passed"])
print("RAW null violations:", raw_report["null_violations"])

cleaned_df, action_log = clean_data(raw_df)
for line in action_log:
    print("-", line)

clean_thresholds = {k: v for k, v in NULL_THRESHOLDS.items() if k != "deck"}
clean_report = run_quality_gate(cleaned_df, SCHEMA, clean_thresholds, DUP_SUBSET)
print("CLEANED gate passed:", clean_report["passed"])

summary = pd.DataFrame({
    "raw_nulls": raw_df.isna().sum(),
    "cleaned_nulls": cleaned_df.reindex(columns=raw_df.columns).isna().sum(),
})
print(summary)
print("rows: raw =", len(raw_df), " cleaned =", len(cleaned_df))
```

**Why this design:** `validate_schema` and `check_null_rates` return *lists of structured violations*, not booleans — an SQA instinct: a test that only says "failed" without saying *what* failed is much less useful than one that hands you the exact broken rows. Age is imputed per `(pclass, sex)` group rather than globally because those two features are strongly correlated with age (e.g. 1st-class passengers skew older), so a group-wise median is a much less biased fill than a single dataset-wide number — this matters directly for model quality downstream, not just cosmetics. The gate is re-run on the *cleaned* output rather than trusted blindly, which is the same principle as re-running your test suite after a fix instead of assuming the fix worked. Note that `duplicate_count` stays at ~135 even after cleaning — this trimmed dataset drops the passenger's name, so several genuinely different passengers share identical values on every remaining column purely by coincidence (common fare, common age, no family aboard). Hard-failing the gate on that would be a false alarm; a real quality gate needs to know which checks are blocking and which are advisory, exactly like triaging bug severity.

</details>